In [3]:
import os
import joblib
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.base import BaseEstimator, TransformerMixin
drive.mount('/content/drive', force_remount=True)

DATA_PATH = '/content/drive/MyDrive/Data/Credit_Risk_Data'

PROJECT_PATH = '/content/drive/MyDrive/Colab Notebooks/Home_Credit_Project'

Mounted at /content/drive


In [4]:
# 1. Основні таблиці
df = pd.read_csv(f'{DATA_PATH}/application_train.csv')
X_test = pd.read_csv(f'{DATA_PATH}/application_test.csv')

y_train = df["TARGET"]
X_train = df.drop(columns=["TARGET"])

# 2. Бюро та попередні заявки
bureau = pd.read_csv(f'{DATA_PATH}/bureau.csv')
bureau_balance = pd.read_csv(f'{DATA_PATH}/bureau_balance.csv')
previous_application = pd.read_csv(f'{DATA_PATH}/previous_application.csv')

# 3. Дочірні таблиці історій платежів
pos_cash_balance = pd.read_csv(f'{DATA_PATH}/POS_CASH_balance.csv')
instalments_payments = pd.read_csv(f'{DATA_PATH}/installments_payments.csv')
credit_card_balance = pd.read_csv(f'{DATA_PATH}/credit_card_balance.csv')

In [5]:
class BureauBalanceFeature(BaseEstimator, TransformerMixin):
    def __init__(self, merge_column="SK_ID_BUREAU"):
        self.merge_column = merge_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df = pd.get_dummies(df, columns=['STATUS'], prefix='STATUS', dtype=float)

        status_cols = [col for col in df.columns if col.startswith('STATUS_')]
        aggregations = {
            'MONTHS_BALANCE': ['min', 'max', 'size']
        }
        for col in status_cols:
            aggregations[col] = ['mean', 'sum']

        df_agg = df.groupby(self.merge_column).agg(aggregations)
        df_agg.columns = [f'BB_{col[0]}_{col[1].upper()}' for col in df_agg.columns]
        df_agg = df_agg.reset_index()

        return df_agg


class BureauFeature(BaseEstimator, TransformerMixin):
    def __init__(self, merge_column="SK_ID_CURR"):
        self.merge_column = merge_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        aggregations = {
            'DAYS_CREDIT': ['min', 'max'],
            'DAYS_CREDIT_ENDDATE': ['min', 'max'],
            'SK_ID_BUREAU': ['count'],
            'AMT_CREDIT_SUM': ['mean', 'max'],
            'AMT_CREDIT_MAX_OVERDUE': ['max'],
        }

        status_bb_col = [col for col in df.columns if col.startswith('BB_STATUS_')]
        for col in status_bb_col:
            aggregations[col] = ['mean', 'max']

        months_bal_bb_col = [col for col in df.columns if col.startswith('BB_MONTHS_BALANCE_')]
        for col in months_bal_bb_col:
            if 'MIN' in col:
                aggregations[col] = ['min']
            elif 'MAX' in col:
                aggregations[col] = ['max']
            elif 'SIZE' in col:
                aggregations[col] = ['sum', 'mean']

        df_agg = df.groupby(self.merge_column).agg(aggregations)
        df_agg.columns = [f'BUREAU_{col[0]}_{col[1].upper()}' for col in df_agg.columns]
        df_agg = df_agg.reset_index()

        return df_agg


class PreviousApplicationFeature(BaseEstimator, TransformerMixin):
    def __init__(self, merge_column="SK_ID_CURR"):
        self.merge_column = merge_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df["PAYMENT_RATE"] = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + 1)
        aggregations = {
            'DAYS_DECISION': ['min', 'max'],
            'PAYMENT_RATE': ['mean', 'max', 'min'],
            'SK_ID_PREV': ['count'],
            'AMT_CREDIT': ['mean', 'max', 'sum'],
            'AMT_ANNUITY': ['mean', 'max'],
            'AMT_DOWN_PAYMENT': ['max', 'mean'],
        }
        df_agg = df.groupby(self.merge_column).agg(aggregations)
        df_agg.columns = [f'PREV_{col[0]}_{col[1].upper()}' for col in df_agg.columns]
        df_agg = df_agg.reset_index()

        return df_agg
class PCBFeature(BaseEstimator, TransformerMixin):
    def __init__(self, merge_column="SK_ID_CURR"):
        self.merge_column = merge_column
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        df = X.copy()
        aggregations = {
        'MONTHS_BALANCE': ['min', 'max', 'size'],
        'SK_DPD': ['max', 'mean'],
        'SK_DPD_DEF': ['max', 'mean'],
        'CNT_INSTALMENT_FUTURE': ['max', 'mean']
        }

        df = pd.get_dummies(df, columns=['NAME_CONTRACT_STATUS'], prefix='NC_STATUS', dtype=float)
        name_contr_stat_col = [col for col in df.columns if col.startswith('NC_STATUS_')]
        for col in name_contr_stat_col:
            aggregations[col] = ['mean']
        df_agg = df.groupby(self.merge_column).agg(aggregations)
        df_agg.columns = [f'PCB_{col[0]}_{col[1].upper()}' for col in df_agg.columns]
        df_agg = df_agg.reset_index()
        return df_agg

class CreditCardFeature(BaseEstimator, TransformerMixin):
    def __init__(self, merge_column="SK_ID_CURR"):
        self.merge_column = merge_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        # Співвідношення боргу до кредитного ліміту (+1 для захисту від ділення на 0)
        df['LIMIT_USAGE'] = df['AMT_BALANCE'] / (df['AMT_CREDIT_LIMIT_ACTUAL'] + 1)

        # Словник агрегацій суто для кредитних карток
        aggregations = {
            'MONTHS_BALANCE': ['min', 'max', 'size'],
            'AMT_BALANCE': ['max', 'mean'],
            'AMT_CREDIT_LIMIT_ACTUAL': ['max', 'mean'],
            'AMT_DRAWINGS_ATM_CURRENT': ['max', 'sum'],
            'AMT_DRAWINGS_CURRENT': ['max', 'sum'],
            'AMT_PAYMENT_CURRENT': ['mean', 'sum'],
            'SK_DPD': ['max', 'mean'],
            'SK_DPD_DEF': ['max', 'mean'],
            'LIMIT_USAGE': ['max', 'mean']
        }

        df = pd.get_dummies(df, columns=['NAME_CONTRACT_STATUS'], prefix='CC_STATUS', dtype=float)
        status_cols = [col for col in df.columns if col.startswith('CC_STATUS_')]
        for col in status_cols:
            aggregations[col] = ['mean']

        df_agg = df.groupby(self.merge_column).agg(aggregations)
        df_agg.columns = [f'CCB_{col[0]}_{col[1].upper()}' for col in df_agg.columns]
        df_agg = df_agg.reset_index()

        return df_agg

class InstallmentsFeature(BaseEstimator, TransformerMixin):
    def __init__(self, merge_column="SK_ID_CURR"):
        self.merge_column = merge_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()

        # платіжних метрик
        delay = df["DAYS_ENTRY_PAYMENT"] - df["DAYS_INSTALMENT"]
        df["DPD"] = delay.clip(lower=0)
        df["PBP"] = (-delay).clip(lower=0)
        #дні затримки
        df["PAYMENT_DIFF"] = df["AMT_INSTALMENT"] - df["AMT_PAYMENT"]
        # скільки дєняг заплатили за кредит і скільки лишилося
        df["PAYMENT_RATIO"] = df["AMT_PAYMENT"] / (df["AMT_INSTALMENT"] + 1)
        # відношення

        aggregations = {
            'DAYS_ENTRY_PAYMENT': ['max'],
            'DPD': ['max', 'sum', 'mean'],
            'PAYMENT_DIFF': ['max', 'mean', 'sum', 'min'],
            'PAYMENT_RATIO': ['mean', 'max'],
            'AMT_INSTALMENT': ['sum', 'mean'],
            'AMT_PAYMENT': ['sum', 'mean']
        }

        df_agg = df.groupby(self.merge_column).agg(aggregations)
        df_agg.columns = [f'INS_{col[0]}_{col[1].upper()}' for col in df_agg.columns]
        df_agg = df_agg.reset_index()

        return df_agg

In [6]:
class FullData(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.ccb_feature = CreditCardFeature()
        self.pcb_feature = PCBFeature()
        self.bureau_feature = BureauFeature()
        self.previous_application_feature = PreviousApplicationFeature()
        self.bb_feature = BureauBalanceFeature()
        self.ins_feature = InstallmentsFeature()
    def fit_transform(self, X, bureau_balance, bureau, credit_card_balance, pos_cash_balance, previous_application, instalments_payments):
        df = X.copy()
        bb_feature = self.bb_feature.transform(bureau_balance)
        bureau_full = bureau.merge(bb_feature, on="SK_ID_BUREAU", how="left")

        ccb_feature = self.ccb_feature.transform(credit_card_balance)
        pcb_feature = self.pcb_feature.transform(pos_cash_balance)
        bureau_feature = self.bureau_feature.transform(bureau_full)
        previous_application_feature = self.previous_application_feature.transform(previous_application)
        ins_feature = self.ins_feature.transform(instalments_payments)


        df_full = df.merge(bureau_feature, on="SK_ID_CURR", how="left")
        df_full = df_full.merge(previous_application_feature, on="SK_ID_CURR", how="left")
        df_full = df_full.merge(pcb_feature, on="SK_ID_CURR", how="left")
        df_full = df_full.merge(ccb_feature, on="SK_ID_CURR", how="left")
        df_full = df_full.merge(ins_feature, on="SK_ID_CURR", how="left")

        return df_full

In [7]:
# Ініціалізація збирача
full_data_builder = FullData()

X_train_full = full_data_builder.fit_transform(
    X=X_train,
    bureau_balance=bureau_balance,
    bureau=bureau,
    credit_card_balance=credit_card_balance,
    pos_cash_balance=pos_cash_balance,
    previous_application=previous_application,
    instalments_payments=instalments_payments
)

X_test_full = full_data_builder.fit_transform(
    X=X_test,
    bureau_balance=bureau_balance,
    bureau=bureau,
    credit_card_balance=credit_card_balance,
    pos_cash_balance=pos_cash_balance,
    previous_application=previous_application,
    instalments_payments=instalments_payments
)

In [8]:
X_train_full = X_train_full.drop(columns=["SK_ID_CURR"])
test_id = X_test_full.pop('SK_ID_CURR')

X_train_full.to_parquet(f'{PROJECT_PATH}/X_train_full.parquet', index=False)
X_test_full.to_parquet(f'{PROJECT_PATH}/X_test_full.parquet', index=False)
y_train.to_frame().to_parquet(f'{PROJECT_PATH}/y_train.parquet', index=False)
test_id.to_frame().to_parquet(f'{PROJECT_PATH}/test_id.parquet', index=False)

print("Усі зібрані таблиці успішно збережено в Home_Credit_Project!")

Усі зібрані таблиці успішно збережено в Home_Credit_Project!
